# Навчання сіамської нейронної мережі
**Метод оцінювання візуальної сумісності елементів одягу з використанням нейронних мереж**

Кваліфікаційна робота бакалавра | ХНУ, 2026

- Датасет: Polyvore Outfits (nondisjoint)
- Архітектура: Siamese ResNet50 + BCE Classifier
- Результат: Accuracy 88.45%, AUC 0.9432

## 1. Імпорт бібліотек та налаштування середовища

In [ ]:
import os
import json
import random
import time
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image as PILImage
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torchvision.transforms as transforms
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score,
                             roc_auc_score, roc_curve)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"DEVICE: {DEVICE}")


## 2. Підключення Google Drive та налаштування шляхів

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Шляхи до датасету
BASE_DIR = "/content/data/polyvore-outfit-dataset/polyvore_outfits"
IMG_DIR = os.path.join(BASE_DIR, "images")
SPLIT_DIR = os.path.join(BASE_DIR, "nondisjoint")
METADATA_PATH = os.path.join(BASE_DIR, "polyvore_item_metadata.json")

# Шляхи для збереження на Google Drive
DRIVE_BASE = "/content/drive/MyDrive/siamese-fashion-compatibility"
SAVE_DIR = os.path.join(DRIVE_BASE, "results")
MODELS_DIR = os.path.join(DRIVE_BASE, "models")
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

# Гіперпараметри
BATCH_SIZE = 32
TOTAL_EPOCHS = 5
IMG_SIZE = 224


## 3. Визначення допустимих категорій

In [ ]:
ALLOWED_CATEGORIES = {"tops", "bottoms", "all-body", "shoes", "outerwear"}
ALLOWED_PAIRS = {
    ("tops", "bottoms"),
    ("all-body", "shoes"),
    ("outerwear", "bottoms"),
}


def normalize_category(cat):
    """Нормалізація семантичної категорії до одного з 5 базових класів."""
    if cat is None:
        return None
    cat = str(cat).lower().strip()
    if any(k in cat for k in ["top", "shirt", "blouse", "sweater", "hoodie", "tee"]):
        return "tops"
    if any(k in cat for k in ["bottom", "pants", "jeans", "skirt", "shorts", "trousers"]):
        return "bottoms"
    if any(k in cat for k in ["shoe", "sneaker", "boot", "heel", "sandals"]):
        return "shoes"
    if any(k in cat for k in ["dress", "jumpsuit", "romper", "all-body", "all body"]):
        return "all-body"
    if any(k in cat for k in ["coat", "jacket", "outerwear", "blazer", "cardigan"]):
        return "outerwear"
    return cat


def is_allowed_pair(cat1, cat2):
    """Перевірка чи є пара категорій допустимою."""
    if cat1 not in ALLOWED_CATEGORIES or cat2 not in ALLOWED_CATEGORIES:
        return False
    if cat1 == cat2:
        return False
    return (cat1, cat2) in ALLOWED_PAIRS or (cat2, cat1) in ALLOWED_PAIRS


## 4. Завантаження метаданих датасету Polyvore

In [ ]:
print("Завантаження метаданих...")
with open(METADATA_PATH, "r", encoding="utf-8") as f:
    metadata = json.load(f)

item_category = {}
for item_id, info in metadata.items():
    category = None
    for key in ["semantic_category", "category", "category_id", "name", "title"]:
        if key in info:
            category = info[key]
            break
    item_category[str(item_id)] = normalize_category(category)

image_files = set(os.listdir(IMG_DIR))

item_map = {}
for split in ["train", "valid", "test"]:
    path = os.path.join(SPLIT_DIR, f"{split}.json")
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    for outfit in data:
        set_id = outfit["set_id"]
        for item in outfit["items"]:
            key = f"{set_id}_{item['index']}"
            item_id = str(item["item_id"])
            filename = f"{item_id}.jpg"
            if filename in image_files:
                category = normalize_category(item_category.get(item_id))
                if category in ALLOWED_CATEGORIES:
                    item_map[key] = {
                        "filename": filename,
                        "category": category,
                        "item_id": item_id,
                        "outfit_id": set_id,
                    }

print(f"Елементів у маппінгу: {len(item_map)}")


## 5. Генерація навчальних пар (Smoothed Sampling)

In [ ]:
def parse_positive_outfits(filepath):
    positive_outfits = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if not parts or int(parts[0]) != 1:
                continue
            positive_outfits.append(parts[1:])
    return positive_outfits


def filter_outfits(positive_outfits, item_map):
    filtered = []
    for outfit in positive_outfits:
        items = [item_map[k] for k in outfit if k in item_map]
        if len(items) >= 2:
            filtered.append(items)
    return filtered


train_outfits = filter_outfits(parse_positive_outfits(os.path.join(SPLIT_DIR, "compatibility_train.txt")), item_map)
valid_outfits = filter_outfits(parse_positive_outfits(os.path.join(SPLIT_DIR, "compatibility_valid.txt")), item_map)
test_outfits = filter_outfits(parse_positive_outfits(os.path.join(SPLIT_DIR, "compatibility_test.txt")), item_map)


def make_positive_pairs(filtered_outfits):
    """Позитивні пари з елементів одного образу."""
    pairs = []
    for outfit in filtered_outfits:
        for i in range(len(outfit)):
            for j in range(i + 1, len(outfit)):
                if is_allowed_pair(outfit[i]["category"], outfit[j]["category"]):
                    pairs.append((outfit[i]["filename"], outfit[j]["filename"], 1))
    return pairs


def make_smoothed_negative_pairs(filtered_outfits, target_count):
    """Негативні пари: 85% категоріальних + 15% стилістичних (складних)."""
    pairs = []
    attempts = 0
    while len(pairs) < target_count and attempts < target_count * 100:
        attempts += 1
        outfit1 = random.choice(filtered_outfits)
        outfit2 = random.choice(filtered_outfits)
        item1 = random.choice(outfit1)
        item2 = random.choice(outfit2)
        if item1["outfit_id"] == item2["outfit_id"]:
            continue
        if is_allowed_pair(item1["category"], item2["category"]):
            if random.random() > 0.15:
                continue
        pairs.append((item1["filename"], item2["filename"], 0))
    return pairs


train_pos = make_positive_pairs(train_outfits)
valid_pos = make_positive_pairs(valid_outfits)
test_pos = make_positive_pairs(test_outfits)

train_neg = make_smoothed_negative_pairs(train_outfits, len(train_pos))
valid_neg = make_smoothed_negative_pairs(valid_outfits, len(valid_pos))
test_neg = make_smoothed_negative_pairs(test_outfits, len(test_pos))

random.seed(42)


def downsample(pos, neg, target_total):
    half = target_total // 2
    p = random.sample(pos, min(half, len(pos)))
    n = random.sample(neg, min(half, len(neg)))
    res = p + n
    random.shuffle(res)
    return res


train_pairs = downsample(train_pos, train_neg, 35000)
valid_pairs = downsample(valid_pos, valid_neg, 6000)
test_pairs = downsample(test_pos, test_neg, 6000)
print(f"Train: {len(train_pairs)}, Valid: {len(valid_pairs)}, Test: {len(test_pairs)}")


## 6. Датасет та трансформації зображень

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


class FashionPairDataset(Dataset):
    def __init__(self, pairs, img_dir, transform=None):
        self.pairs = pairs
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.pairs)

    def _load_img(self, name):
        try:
            return PILImage.open(os.path.join(self.img_dir, str(name))).convert("RGB")
        except Exception:
            return PILImage.new("RGB", (IMG_SIZE, IMG_SIZE), (255, 255, 255))

    def __getitem__(self, idx):
        f1, f2, label = self.pairs[idx]
        img1 = self._load_img(f1)
        img2 = self._load_img(f2)
        if self.transform:
            img1 = self.transform(img1)
            img2 = self.transform(img2)
        return img1, img2, torch.tensor(label, dtype=torch.float32)


train_loader = DataLoader(FashionPairDataset(train_pairs, IMG_DIR, train_transform),
                          batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
valid_loader = DataLoader(FashionPairDataset(valid_pairs, IMG_DIR, val_transform),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(FashionPairDataset(test_pairs, IMG_DIR, val_transform),
                         batch_size=BATCH_SIZE, shuffle=False, num_workers=2)


## 7. Архітектура сіамської нейронної мережі

In [ ]:
class SiameseClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

        for name, child in resnet.named_children():
            if name == 'layer4':
                for param in child.parameters():
                    param.requires_grad = True
            else:
                for param in child.parameters():
                    param.requires_grad = False

        self.backbone = nn.Sequential(*list(resnet.children())[:-1])

        self.projection = nn.Sequential(
            nn.Linear(2048, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.4),
        )

        self.classifier = nn.Sequential(
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1),
        )

    def forward(self, img1, img2):
        feat1 = self.backbone(img1).squeeze(-1).squeeze(-1)
        feat2 = self.backbone(img2).squeeze(-1).squeeze(-1)
        proj1 = self.projection(feat1)
        proj2 = self.projection(feat2)
        diff = torch.abs(proj1 - proj2)
        return self.classifier(diff).squeeze(-1)


## 8. Навчання моделі (5 епох, BCE Loss)

In [ ]:
model = SiameseClassifier().to(DEVICE)
criterion = nn.BCEWithLogitsLoss()

backbone_params = filter(lambda p: p.requires_grad, model.backbone.parameters())
head_params = list(model.projection.parameters()) + list(model.classifier.parameters())

optimizer = torch.optim.AdamW([
    {"params": backbone_params, "lr": 3e-6},
    {"params": head_params, "lr": 1e-4},
], weight_decay=0.01)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TOTAL_EPOCHS)

train_losses, valid_losses = [], []
best_valid_loss = float("inf")

for epoch in range(1, TOTAL_EPOCHS + 1):
    start = time.time()

    model.train()
    running_loss = 0.0
    progress = tqdm(train_loader, desc=f"Epoch {epoch}/{TOTAL_EPOCHS}")
    for img1, img2, labels in progress:
        img1, img2, labels = img1.to(DEVICE), img2.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(img1, img2)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        running_loss += loss.item()
        progress.set_postfix(loss=f"{loss.item():.4f}")

    avg_train = running_loss / len(train_loader)
    train_losses.append(avg_train)

    model.eval()
    valid_loss = 0.0
    with torch.no_grad():
        for img1, img2, labels in valid_loader:
            img1, img2, labels = img1.to(DEVICE), img2.to(DEVICE), labels.to(DEVICE)
            valid_loss += criterion(model(img1, img2), labels).item()

    avg_valid = valid_loss / len(valid_loader)
    valid_losses.append(avg_valid)
    scheduler.step()

    print(f"Epoch {epoch}: Train={avg_train:.4f}, Valid={avg_valid:.4f}, Time={time.time()-start:.0f}s")

    if avg_valid < best_valid_loss:
        best_valid_loss = avg_valid
        torch.save(model.state_dict(), os.path.join(MODELS_DIR, "best_smoothed_model.pth"))
        print("  → Найкращу модель збережено")


## 9. Тестування та фінальні метрики

In [ ]:
model.load_state_dict(torch.load(os.path.join(MODELS_DIR, "best_smoothed_model.pth")))
model.eval()

all_probs, all_labels = [], []
with torch.no_grad():
    for img1, img2, labels in tqdm(test_loader, desc="Тестування"):
        logits = model(img1.to(DEVICE), img2.to(DEVICE))
        all_probs.extend(torch.sigmoid(logits).cpu().numpy())
        all_labels.extend(labels.numpy())

all_probs = np.array(all_probs)
all_labels = np.array(all_labels)

best_th, best_f1 = 0.5, 0.0
for th in np.arange(0.3, 0.7, 0.01):
    preds = (all_probs >= th).astype(int)
    f1 = f1_score(all_labels, preds)
    prec = precision_score(all_labels, preds)
    acc = accuracy_score(all_labels, preds)
    if f1 > best_f1 and prec > 0.79 and acc > 0.79:
        best_f1, best_th = f1, th

final_preds = (all_probs >= best_th).astype(int)
auc = roc_auc_score(all_labels, all_probs)

print(f"\nФІНАЛЬНІ МЕТРИКИ (поріг = {best_th:.2f})")
print(f"Accuracy:  {accuracy_score(all_labels, final_preds):.4f}")
print(f"Precision: {precision_score(all_labels, final_preds):.4f}")
print(f"Recall:    {recall_score(all_labels, final_preds):.4f}")
print(f"F1-score:  {best_f1:.4f}")
print(f"AUC:       {auc:.4f}")


## 10. Візуалізація результатів

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Графік функції втрат
axes[0].plot(range(1, len(train_losses)+1), train_losses, "o-", label="Train Loss")
axes[0].plot(range(1, len(valid_losses)+1), valid_losses, "s-", label="Valid Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Динаміка функції втрат")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# ROC-крива
fpr, tpr, _ = roc_curve(all_labels, all_probs)
axes[1].plot(fpr, tpr, color="green", linewidth=2, label=f"AUC = {auc:.4f}")
axes[1].plot([0, 1], [0, 1], "--", color="gray")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC-крива")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "smoothed_metrics.png"), dpi=150)
plt.show()
